CBAMが想定道理になっているか，ちゃんと効果があるかなどを検証

In [ ]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth # DropPath用
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from collections import OrderedDict

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp_clean/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp_clean/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_custom_cnn"

# ★ フラグ設定
USE_SCHEDULER = False # Trueでコサイン波状に学習率を落とす、Falseで固定
CNN_INNER_DROPOUT = 0.2 # ★ NEW: ConvNeXt内部のブロックごとに適用するDropout確率

# --- 1. カスタム ConvNeXt 実装 (完全互換版) ---

# ConvNeXt特有のチャンネル次元LayerNorm
class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

# 次元入れ替え用ヘルパー
class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

# ConvNeXtの基本ブロック (完全互換版・修正済)
class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        # OrderedDict を使って、公式と同じレイヤー番号(名前)を強制的に付ける
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)), # Depthwise
            ("1", Permute([0, 2, 3, 1])),                # チャンネルを一番最後に移動
            ("2", nn.LayerNorm(dim, eps=1e-6)),          # LayerNorm
            ("3", nn.Linear(dim, 4 * dim, bias=True)),   # 膨張
            ("4", nn.GELU()),                            # 活性化
            ("custom_drop", nn.Dropout(p=dropout_p)),    # ★ カスタムDropout
            ("5", nn.Linear(4 * dim, dim, bias=True)),   # 収縮
        ]))
        
        # ★ 修正1: 公式の重みデータと同じ [dim, 1, 1] の形に戻す
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        
        # ★ 修正2: layer_scaleを掛ける「前」に元の画像形式 [B, C, H, W] に戻す
        result = result.permute(0, 3, 1, 2)
        
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input # 残差接続
        return result

# 全体の特徴抽出器を構築する関数
def create_custom_convnext_features(pretrained=True, dropout_p=0.3):
    # ConvNeXt-Tinyの公式仕様
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    
    features = nn.Sequential()
    
    # 0. Stem層
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    
    # 1. 各Stageの構築
    for i in range(4):
        # Stage間のダウンサンプリング
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        
        # ConvNeXtブロックの積み重ね
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))

    # 事前学習済み重みのロード
    if pretrained:
        print("Loading official ConvNeXt-Tiny pre-trained weights...")
        orig_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        # 構造を完全に合わせたので、エラーなくスッと重みが入り、custom_dropだけが無視されます
        features.load_state_dict(orig_model.features.state_dict(), strict=False)
        
    return features


# --- 2. CBAM モジュール ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        return ch_att * sp_att

# --- 3. 融合モデル ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        
        # ★ ここをカスタムConvNeXtに差し替え
        self.dl_extractor = create_custom_convnext_features(pretrained=True, dropout_p=CNN_INNER_DROPOUT)
        dl_out_channels = 768
        
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        
        return self.classifier(fused)

# --- 4. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 5. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 6. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 16
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

# 訓練用 (データ拡張あり)
    train_transform = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        
        # ★ NEW: カラージッターを追加
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=train_transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=val_transform, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        print("Scheduler: ENABLED (Cosine Annealing)")
    else:
        scheduler = None
        print("Scheduler: DISABLED (Fixed Learning Rate)")
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

CBAM→1Dアテンションゲート

In [4]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth # DropPath用
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from collections import OrderedDict

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_custom_cnn"

# ★ フラグ設定
USE_SCHEDULER = False # Trueでコサイン波状に学習率を落とす、Falseで固定
CNN_INNER_DROPOUT = 0.2 # ★ NEW: ConvNeXt内部のブロックごとに適用するDropout確率

# --- 1. カスタム ConvNeXt 実装 (完全互換版) ---

# ConvNeXt特有のチャンネル次元LayerNorm
class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

# 次元入れ替え用ヘルパー
class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

# ConvNeXtの基本ブロック (完全互換版・修正済)
class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        # OrderedDict を使って、公式と同じレイヤー番号(名前)を強制的に付ける
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)), # Depthwise
            ("1", Permute([0, 2, 3, 1])),                # チャンネルを一番最後に移動
            ("2", nn.LayerNorm(dim, eps=1e-6)),          # LayerNorm
            ("3", nn.Linear(dim, 4 * dim, bias=True)),   # 膨張
            ("4", nn.GELU()),                            # 活性化
            ("custom_drop", nn.Dropout(p=dropout_p)),    # ★ カスタムDropout
            ("5", nn.Linear(4 * dim, dim, bias=True)),   # 収縮
        ]))
        
        # ★ 修正1: 公式の重みデータと同じ [dim, 1, 1] の形に戻す
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        
        # ★ 修正2: layer_scaleを掛ける「前」に元の画像形式 [B, C, H, W] に戻す
        result = result.permute(0, 3, 1, 2)
        
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input # 残差接続
        return result

# 全体の特徴抽出器を構築する関数
def create_custom_convnext_features(pretrained=True, dropout_p=0.3):
    # ConvNeXt-Tinyの公式仕様
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    
    features = nn.Sequential()
    
    # 0. Stem層
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    
    # 1. 各Stageの構築
    for i in range(4):
        # Stage間のダウンサンプリング
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        
        # ConvNeXtブロックの積み重ね
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))

    # 事前学習済み重みのロード
    if pretrained:
        print("Loading official ConvNeXt-Tiny pre-trained weights...")
        orig_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        # 構造を完全に合わせたので、エラーなくスッと重みが入り、custom_dropだけが無視されます
        features.load_state_dict(orig_model.features.state_dict(), strict=False)
        
    return features


# --- 2. 1Dアテンションゲート (Late Fusion用) ---
class ChannelAttentionGate1D(nn.Module):
    """
    1次元ベクトル同士を融合するためのアテンションゲート。
    画像(Y)と数値データ(X)の「どのチャネルを重視するか」を動的に決定します。
    """
    def __init__(self, channels, reduction=16, initial_prob=0.5):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        
        # 空間次元(H, W)がないため、シンプルな全結合層(MLP)で重要度を計算
        self.attention = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid = nn.Sigmoid()
        self._initialize_weights(initial_prob)

    def _initialize_weights(self, target_prob):
        # 初期状態で出力が target_prob (例: 0.5) になるようにバイアスを調整
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.attention[2].bias, logit_value)
            self.attention[2].weight.data *= 0.01

    def forward(self, x, y):
        # x: 数値データ特徴量 [B, 768]
        # y: 画像特徴量 [B, 768]
        
        context = x + y
        weights = self.sigmoid(self.attention(context)) # [B, 768]
        
        # 計算された重みで加重平均をとる
        fused = weights * x + (1 - weights) * y
        return fused


# --- 3. 融合モデル (Late Fusion版) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        
        dl_out_channels = 768
        
        # --- 画像側 (Y) ---
        self.dl_extractor = create_custom_convnext_features(pretrained=True, dropout_p=CNN_INNER_DROPOUT)
        
        # 画像の空間情報(H, W)を1点に集約して1Dベクトルにする
        self.dl_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels)
        )
        
        # --- 数値データ側 (X) ---
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        # --- 融合モジュール ---
        self.fusion_gate = ChannelAttentionGate1D(dl_out_channels, initial_prob=0.5) 
        
        # --- 最終分類器 ---
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        # 1. 画像特徴量の抽出と1次元化
        img_feat = self.dl_extractor(img)          # [B, 768, H, W]
        img_feat_1d = self.dl_pool(img_feat)       # [B, 768]
        
        # 2. 数値データの投影
        hc_feat_1d = self.hc_projector(hc_vec)     # [B, 768]
        
        # 3. アテンションによる1次元空間での融合
        fused_feat = self.fusion_gate(hc_feat_1d, img_feat_1d) # [B, 768]
        
        # 4. 分類
        out = self.classifier(fused_feat)          # [B, num_classes]
        return out


# --- 4. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: image = Image.open(img_path).convert('RGB')
        except: image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform: image = self.transform(image)
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# --- 5. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 6. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 8
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    # 訓練用 (データ拡張あり)
    train_transform = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=train_transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=val_transform, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        print("Scheduler: ENABLED (Cosine Annealing)")
    else:
        scheduler = None
        print("Scheduler: DISABLED (Fixed Learning Rate)")
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Loading official ConvNeXt-Tiny pre-trained weights...
Scheduler: DISABLED (Fixed Learning Rate)
Starting Training on cuda...
Epoch 1/100 | Train Loss: 1.0870 | Val Loss: 1.0154 | Val Acc: 0.4375 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.4375)
Epoch 2/100 | Train Loss: 0.9967 | Val Loss: 0.9253 | Val Acc: 0.5625 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.5625)
Epoch 3/100 | Train Loss: 0.9681 | Val Loss: 0.9472 | Val Acc: 0.5417 | LR: 0.000100
Epoch 4/100 | Train Loss: 0.9329 | Val Loss: 1.0157 | Val Acc: 0.5694 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.5694)
Epoch 5/100 | Train Loss: 0.9229 | Val Loss: 0.9338 | Val Acc: 0.5556 | LR: 0.000100
Epoch 6/100 | Train Loss: 0.8886 | Val Loss: 1.0596 | Val Acc: 0.6319 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.6319)
Epoch 7/100 | Train Loss: 0.9110 | Val Loss: 0.8816 | Val Acc: 0.5833 | LR: 0.000100
Epoch 8/100 | Train Loss: 0.8995 | Val Loss: 0.8626 | Val Acc: 0.6250 | LR: 0.000100
Epoch 9/100 | Train Loss: 0.8975 | Val Loss

CNNで使用するデータ直し

In [3]:
from sklearn.preprocessing import StandardScaler
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth # DropPath用
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from collections import OrderedDict

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_custom_cnn"

# ★ フラグ設定
USE_SCHEDULER = False # Trueでコサイン波状に学習率を落とす、Falseで固定
CNN_INNER_DROPOUT = 0.2 # ★ NEW: ConvNeXt内部のブロックごとに適用するDropout確率

# --- 1. カスタム ConvNeXt 実装 (完全互換版) ---

# ConvNeXt特有のチャンネル次元LayerNorm
class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

# 次元入れ替え用ヘルパー
class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

# ConvNeXtの基本ブロック (完全互換版・修正済)
class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        # OrderedDict を使って、公式と同じレイヤー番号(名前)を強制的に付ける
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)), # Depthwise
            ("1", Permute([0, 2, 3, 1])),                # チャンネルを一番最後に移動
            ("2", nn.LayerNorm(dim, eps=1e-6)),          # LayerNorm
            ("3", nn.Linear(dim, 4 * dim, bias=True)),   # 膨張
            ("4", nn.GELU()),                            # 活性化
            ("custom_drop", nn.Dropout(p=dropout_p)),    # ★ カスタムDropout
            ("5", nn.Linear(4 * dim, dim, bias=True)),   # 収縮
        ]))
        
        # ★ 修正1: 公式の重みデータと同じ [dim, 1, 1] の形に戻す
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        
        # ★ 修正2: layer_scaleを掛ける「前」に元の画像形式 [B, C, H, W] に戻す
        result = result.permute(0, 3, 1, 2)
        
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input # 残差接続
        return result

# 全体の特徴抽出器を構築する関数
def create_custom_convnext_features(pretrained=True, dropout_p=0.3):
    # ConvNeXt-Tinyの公式仕様
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    
    features = nn.Sequential()
    
    # 0. Stem層
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    
    # 1. 各Stageの構築
    for i in range(4):
        # Stage間のダウンサンプリング
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        
        # ConvNeXtブロックの積み重ね
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))

    # 事前学習済み重みのロード
    if pretrained:
        print("Loading official ConvNeXt-Tiny pre-trained weights...")
        orig_model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        # 構造を完全に合わせたので、エラーなくスッと重みが入り、custom_dropだけが無視されます
        features.load_state_dict(orig_model.features.state_dict(), strict=False)
        
    return features


# --- 2. 1Dアテンションゲート (Late Fusion用) ---
class ChannelAttentionGate1D(nn.Module):
    """
    1次元ベクトル同士を融合するためのアテンションゲート。
    画像(Y)と数値データ(X)の「どのチャネルを重視するか」を動的に決定します。
    """
    def __init__(self, channels, reduction=16, initial_prob=0.5):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        
        # 空間次元(H, W)がないため、シンプルな全結合層(MLP)で重要度を計算
        self.attention = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid = nn.Sigmoid()
        self._initialize_weights(initial_prob)

    def _initialize_weights(self, target_prob):
        # 初期状態で出力が target_prob (例: 0.5) になるようにバイアスを調整
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.attention[2].bias, logit_value)
            self.attention[2].weight.data *= 0.01

    def forward(self, x, y):
        # x: 数値データ特徴量 [B, 768]
        # y: 画像特徴量 [B, 768]
        
        context = x + y
        weights = self.sigmoid(self.attention(context)) # [B, 768]
        
        # 計算された重みで加重平均をとる
        fused = weights * x + (1 - weights) * y
        return fused


# --- 3. 融合モデル (Late Fusion版) ---
class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        
        dl_out_channels = 768
        
        # --- 画像側 (Y) ---
        self.dl_extractor = create_custom_convnext_features(pretrained=True, dropout_p=CNN_INNER_DROPOUT)
        
        # 画像の空間情報(H, W)を1点に集約して1Dベクトルにする
        self.dl_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels)
        )
        
        # --- 数値データ側 (X) ---
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        # --- 融合モジュール ---
        self.fusion_gate = ChannelAttentionGate1D(dl_out_channels, initial_prob=0.5) 
        
        # --- 最終分類器 ---
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        # 1. 画像特徴量の抽出と1次元化
        img_feat = self.dl_extractor(img)          # [B, 768, H, W]
        img_feat_1d = self.dl_pool(img_feat)       # [B, 768]
        
        # 2. 数値データの投影
        hc_feat_1d = self.hc_projector(hc_vec)     # [B, 768]
        
        # 3. アテンションによる1次元空間での融合
        fused_feat = self.fusion_gate(hc_feat_1d, img_feat_1d) # [B, 768]
        
        # 4. 分類
        out = self.classifier(fused_feat)          # [B, num_classes]
        return out


# --- 4. Datasetクラス ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # ★ CSVのファイル名にある '_mask' を '_combined' に置換
        original_filename = row['filename']
        actual_filename = original_filename.replace('_mask', '_combined')
        
        # ★ 'mask' フォルダではなく 'combined' フォルダを参照
        img_path = os.path.join(self.img_root, 'combined', row['category'], actual_filename)
        
        try: 
            image = Image.open(img_path).convert('RGB')
        except Exception as e: 
            # 読み込み失敗時の確認用にエラーを出力しておくとデバッグしやすいです
            # print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return image, hc_feat, label

# --- 5. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 6. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 16
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_fusion_model.pth'

    # 訓練用 (データ拡張あり)
    train_transform = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=train_transform, is_train=True)
    val_ds = CustomMultiModalDataset(val_df, IMG_ROOT, scaler=train_ds.scaler, transform=val_transform, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = FusionModel(num_classes=3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        print("Scheduler: ENABLED (Cosine Annealing)")
    else:
        scheduler = None
        print("Scheduler: DISABLED (Fixed Learning Rate)")
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        for images, hc_feats, labels in train_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images, hc_feats), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, hc_feats, labels in val_loader:
                images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
                outputs = model(images, hc_feats)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Loading official ConvNeXt-Tiny pre-trained weights...
Scheduler: DISABLED (Fixed Learning Rate)
Starting Training on cuda...
Epoch 1/100 | Train Loss: 0.5291 | Val Loss: 0.2456 | Val Acc: 0.9236 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.9236)
Epoch 2/100 | Train Loss: 0.2464 | Val Loss: 0.3270 | Val Acc: 0.8264 | LR: 0.000100
Epoch 3/100 | Train Loss: 0.1625 | Val Loss: 0.3076 | Val Acc: 0.8542 | LR: 0.000100
Epoch 4/100 | Train Loss: 0.0894 | Val Loss: 0.7137 | Val Acc: 0.7847 | LR: 0.000100
Epoch 5/100 | Train Loss: 0.0857 | Val Loss: 0.1722 | Val Acc: 0.9444 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.9444)
Epoch 6/100 | Train Loss: 0.0542 | Val Loss: 0.6889 | Val Acc: 0.8125 | LR: 0.000100
Epoch 7/100 | Train Loss: 0.0473 | Val Loss: 0.4212 | Val Acc: 0.8889 | LR: 0.000100
Epoch 8/100 | Train Loss: 0.0512 | Val Loss: 0.2530 | Val Acc: 0.9028 | LR: 0.000100
Epoch 9/100 | Train Loss: 0.0555 | Val Loss: 0.2711 | Val Acc: 0.8889 | LR: 0.000100
Epoch 10/100 | Train Loss: 0.0513 

In [7]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import math
from collections import OrderedDict

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

# スケーラーを訓練時と同じ状態にするため、Train用のCSVも必要
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'

# 学習時に合わせた設定
TARGET_EPOCH = 100
CNN_INNER_DROPOUT = 0.2
SAVE_DIR = f"../save/CBAMtest/nomal_ConvNext_{TARGET_EPOCH}epoch_custom_cnn"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_fusion_model.pth'


# --- 2. モデルの定義 (ロード用に訓練スクリプトと同じものを定義) ---

class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)),
            ("1", Permute([0, 2, 3, 1])),
            ("2", nn.LayerNorm(dim, eps=1e-6)),
            ("3", nn.Linear(dim, 4 * dim, bias=True)),
            ("4", nn.GELU()),
            ("custom_drop", nn.Dropout(p=dropout_p)),
            ("5", nn.Linear(4 * dim, dim, bias=True)),
        ]))
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        result = result.permute(0, 3, 1, 2)
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input
        return result

def create_custom_convnext_features(pretrained=False, dropout_p=0.3):
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    features = nn.Sequential()
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    for i in range(4):
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))
    return features

class ChannelAttentionGate1D(nn.Module):
    def __init__(self, channels, reduction=16, initial_prob=0.5):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        self.attention = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid = nn.Sigmoid()
        self._initialize_weights(initial_prob)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.attention[2].bias, logit_value)
            self.attention[2].weight.data *= 0.01

    def forward(self, x, y):
        context = x + y
        weights = self.sigmoid(self.attention(context))
        return weights * x + (1 - weights) * y

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        dl_out_channels = 768
        
        self.dl_extractor = create_custom_convnext_features(pretrained=False, dropout_p=CNN_INNER_DROPOUT)
        self.dl_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels)
        )
        
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        self.fusion_gate = ChannelAttentionGate1D(dl_out_channels, initial_prob=0.5) 
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        img_feat = self.dl_extractor(img)
        img_feat_1d = self.dl_pool(img_feat)
        hc_feat_1d = self.hc_projector(hc_vec)
        fused_feat = self.fusion_gate(hc_feat_1d, img_feat_1d)
        return self.classifier(fused_feat)


# --- 3. Datasetの定義 (マルチモーダル版) ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        # 評価用なので、渡された学習済みscalerでtransformのみ行う
        features = self.df[self.feature_cols].values
        self.hc_features = scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # ファイル名の '_mask' を '_combined' に置換
        original_filename = row['filename']
        actual_filename = original_filename.replace('_mask', '_combined')
        
        # 'combined' フォルダを参照
        img_path = os.path.join(self.img_root, 'combined', row['category'], actual_filename)
        
        try: 
            image = Image.open(img_path).convert('RGB')
        except Exception as e: 
            print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return image, hc_feat, label


# --- 4. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}\n先に学習を完了させてください。")
        return

    print("=== Starting Test Evaluation for Late Fusion Model ===")

    # 1. 訓練時のスケーラーを再現
    print("Fitting StandardScaler on Training data...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols].values)

    # 2. テストデータセットの準備 (Validation時と同じ224x224のリサイズを使用)
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=scaler, transform=test_transform)
    test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

    # 3. モデルの準備と重みのロード
    model = FusionModel(num_classes=3).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE))
    model.eval()
    print(f"Successfully loaded weights from {MODEL_WEIGHT_PATH}")

    all_preds, all_labels = [], []
    
    # 4. 推論ループ
    print("Running inference on test dataset...")
    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 5. 結果の算出と表示
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\nOverall Test Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 6. 混同行列の保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Late Fusion Model)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    cm_save_path = f'{SAVE_DIR}/test_confusion_matrix.png'
    plt.savefig(cm_save_path)
    plt.close()
    
    print(f"Confusion matrix saved to {cm_save_path}")
    print("Test Evaluation Finished.")

if __name__ == "__main__":
    evaluate_model()

=== Starting Test Evaluation for Late Fusion Model ===
Fitting StandardScaler on Training data...
Successfully loaded weights from ../save/CBAMtest/nomal_ConvNext_100epoch_custom_cnn/best_fusion_model.pth
Running inference on test dataset...

Overall Test Accuracy: 0.9514

Classification Report:
              precision    recall  f1-score   support

           0     0.9796    1.0000    0.9897        48
           1     0.9184    0.9375    0.9278        48
           2     0.9565    0.9167    0.9362        48

    accuracy                         0.9514       144
   macro avg     0.9515    0.9514    0.9512       144
weighted avg     0.9515    0.9514    0.9512       144

Confusion matrix saved to ../save/CBAMtest/nomal_ConvNext_100epoch_custom_cnn/test_confusion_matrix.png
Test Evaluation Finished.
